## Training

This notebook allows training new models and creating new versions of the Amazon dataset (with different hyperparameters).

If the dataset already exists (see the *data* folder), it will not be created again, only loaded.

### 1. Creating or loading the dataset

In [ ]:
from src.datasets.text_datasets.AmazonDataset import AmazonDataset

# Dataset Configuration. 
# This configuration is used to create a dataset that can be applied to any model (BOW, ATT, ...). Therefore, it is quite complex.

dataset = "amazon"
subset = "fashion"

seed = 100
min_reviews_rst = 100
min_reviews_usr = 1
bow_pct_words = 10

remove_stopwords = 2 # 0, 1 or 2 (Do not remove, manual removal, automatic removal)
lemmatization = True
remove_accents = True
remove_numbers = True
truncate_padding = True

dts_cfg = {"dataset": dataset, "subset": subset, "seed": seed, "save_path": "data/",
            "remove_stopwords": remove_stopwords, "remove_accents": remove_accents, "remove_numbers": remove_numbers,
            "lemmatization": lemmatization,
            "min_reviews_rst": min_reviews_rst, "min_reviews_usr": min_reviews_usr,
            "min_df": 5, "bow_pct_words": bow_pct_words, "presencia": False, "text_column": "text",
            "n_max_words": -50, "test_dev_split": .1, "truncate_padding": truncate_padding}

# Crear el objeto dataset
text_dataset = None

if dataset == "amazon":
    # Location of the original downloaded data; in our case, they are in this folder.  
    # Downloaded them from here (2018 version): https://cseweb.ucsd.edu/~jmcauley/datasets/amazon_v2/
    dts_cfg["data_path"] = "/media/nas/datasets/amazon/"
    # Review language (for POS)
    dts_cfg["language"] = "en"
    # Create the object. This automatically creates a folder in dts_cfg["save_path"] with the dataset (if it does not already exist)
    text_dataset = AmazonDataset(dts_cfg)
else:
    raise ValueError

### 2. Create model and train

In [ ]:
from src.models.text_models.att.ATT2ITM import ATT2ITM
from src.models.text_models.BOW2ITM import BOW2ITM
import numpy as np
import nvgpu

model_name = "BOW2ITM"
model_v = "0" # If there are many versions of the same model (architectures)

l_rate = 5e-05
n_epochs = 1000 # Maximum number of epochs. It could be lower if early stopping is triggered.
b_size = 1024

gpu = np.argmin([g["mem_used_percent"] for g in nvgpu.gpu_info()]) # By default, if multiple are available, the least occupied GPU is selected

model_config = {"model": {"model_version": model_v, "learning_rate": l_rate, "final_learning_rate": l_rate/100, "epochs": n_epochs, "batch_size": b_size, "seed": seed,
                          "early_st_first_epoch": 0, "early_st_monitor": "val_loss", "early_st_monitor_mode": "min", "early_st_patience": 50},
                "session": {"gpu": gpu, "mixed_precision": False, "in_md5": False}}

# Create model object
model = None

if "BOW2ITM" == model_name:
    model = BOW2ITM(model_config, text_dataset)
elif "ATT2ITM" == model_name:
    model = ATT2ITM(model_config, text_dataset)
else:
    raise ValueError

#### 2.1 Test hyperparameters without saving the model

In [ ]:
model.train(dev=True, save_model=False)

#### 2.3 Test hyperparameters saving the model

In [ ]:
model.train(dev=True, save_model=True)

#### 2.1 Final model training 
Training the best number of epochs using train+val

In [ ]:
model.train(dev=False, save_model=True)